# Chapter 07 — SOLUTIONS: Introduction to Unsupervised Learning

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

digits = load_digits()
X = digits.data
y = digits.target
print(f'Digits: {X.shape[0]} samples, {X.shape[1]} features')

## Task 1 Solution: PCA 2D Projection

In [ ]:
# 1a & 1b: Scale and apply PCA
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

var_explained = pca.explained_variance_ratio_.sum()
print(f'Reduced from {X.shape[1]} to 2 dimensions')
print(f'Variance retained: {var_explained:.1%}')

In [ ]:
# 1c: Unlabeled view
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_2d[:, 0], X_2d[:, 1], color='steelblue', alpha=0.3, s=15)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('Digits: 64D → 2D via PCA (no labels)', fontsize=13)
plt.tight_layout()
plt.show()
print('Can you see roughly 10 groups? Some are clearly separate, others overlap.')

In [ ]:
# 1d: Labeled view
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y,
                     cmap='tab10', alpha=0.5, s=15)
plt.colorbar(scatter, ax=ax, label='Digit')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('Digits 2D PCA — True Labels Revealed', fontsize=13)
plt.tight_layout()
plt.show()

print('Observations:')
print('→ Digit 0 (purple) and 4 tend to form clean clusters')
print('→ Digits 3, 5, 8 overlap more — they look similar in pixel space')
print(f'→ Only {var_explained:.1%} variance captured — many groups still visible!')

## Task 2 Solution: Clustering Confirmation

In [ ]:
# 2a: K-Means with k=10
kmeans = KMeans(n_clusters=10, n_init=20, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# 2b: Adjusted Rand Index
ari = adjusted_rand_score(y, cluster_labels)
print(f'Adjusted Rand Index: {ari:.4f}')
print('(1.0 = perfect match with true digit labels)')
print()
print('K-Means recovers the digit structure without ever seeing any labels!')
print('The ARI > 0 confirms that the discovered clusters strongly align with true digits.')

# Visualize in 2D
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='tab10', alpha=0.5, s=15)
axes[0].set_title(f'K-Means clusters (k=10)\nARI = {ari:.3f}', fontsize=12)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', alpha=0.5, s=15)
axes[1].set_title('True digit labels', fontsize=12)
for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
plt.suptitle('K-Means vs True Labels (note: cluster colors may differ!)', fontsize=13)
plt.tight_layout()
plt.show()

## Bonus Solution: How Much Variance Does PCA Capture?

In [ ]:
pca_full = PCA().fit(X_scaled)
cumvar = pca_full.explained_variance_ratio_.cumsum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='#3498db', linewidth=2, markersize=4)

for thresh, color in [(0.80, '#2ecc71'), (0.95, '#e74c3c')]:
    n_comp = np.argmax(cumvar >= thresh) + 1
    ax.axhline(thresh, color=color, linestyle='--', alpha=0.7)
    ax.annotate(f'{thresh:.0%}: {n_comp} components',
                xy=(n_comp, thresh), xytext=(n_comp + 1, thresh - 0.07),
                fontsize=10, color=color)
    print(f'{thresh:.0%} variance → {n_comp} components')

ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA Scree Plot — Digits Dataset (64 features)', fontsize=13)
plt.tight_layout()
plt.show()

print('\n→ We only need ~20 components for 95% of the variance.')
print('   This is a massive compression from 64 features!')
print('   We will master this in Ch09 — Dimensionality Reduction.')